In [28]:
import pandas as pd


data = pd.read_csv(
"../data/processed/final_aml_dataset.csv"
)


data.head()

,txId,time,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,class,transaction_velocity,night_transaction,high_amount_flag,anomaly,is_suspicious,risk_score,risk_level,degree,centrality
0,0,-1.556878,0.307025,1.168821,-0.748109,7.643807,-0.053406,6.039024,21.528480,-0.146334,...,2,2147,0,0,1,False,30,LOW,NaN,NaN
1,1,-1.556878,0.067780,0.271911,-0.226081,2.628539,-0.053406,2.820415,1.501088,-0.146334,...,2,2147,0,0,1,False,30,LOW,NaN,NaN
2,2,-1.556878,-0.135517,-0.222646,-1.270138,-0.183761,-0.041930,-0.187137,-0.080022,-0.102664,...,2,2147,0,0,1,False,30,LOW,NaN,NaN
3,3,-1.556878,-0.140505,-0.222646,-1.270138,-0.183761,-0.041930,-0.187137,-0.080022,-0.108753,...,2,2147,0,0,1,False,30,LOW,NaN,NaN
4,4,-1.556878,-0.170324,-0.222646,-1.270138,-0.090018,-0.041930,-0.134373,0.447015,-0.146324,...,2,2147,0,0,1,False,30,LOW,NaN,NaN


In [29]:
import joblib


model = joblib.load(
"../models/random_forest_fraud_model.pkl"
)

In [33]:
model.feature_names_in_

array(['txId', 'time', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9',
       'feature_10', 'feature_11', 'feature_12', 'feature_13',
       'feature_14', 'feature_15', 'feature_16', 'feature_17',
       'feature_18', 'feature_19', 'feature_20', 'feature_21',
       'feature_22', 'feature_23', 'feature_24', 'feature_25',
       'feature_26', 'feature_27', 'feature_28', 'feature_29',
       'feature_30', 'feature_31', 'feature_32', 'feature_33',
       'feature_34', 'feature_35', 'feature_36', 'feature_37',
       'feature_38', 'feature_39', 'feature_40', 'feature_41',
       'feature_42', 'feature_43', 'feature_44', 'feature_45',
       'feature_46', 'feature_47', 'feature_48', 'feature_49',
       'feature_50', 'feature_51', 'feature_52', 'feature_53',
       'feature_54', 'feature_55', 'feature_56', 'feature_57',
       'feature_58', 'feature_59', 'feature_60', 'feature_61',
       'feature_62', 'feature_63',

In [36]:
# Select only features used during training

features_used = model.feature_names_in_


X = data[
    features_used
]


# Get probabilities

probabilities = model.predict_proba(
    X
)


# Find fraud class probability

fraud_index = list(
    model.classes_
).index(1)


data["ml_probability"] = (
    probabilities[:, fraud_index] * 100
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_20720\2453480748.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["ml_probability"] = (


In [37]:
data["anomaly_score"] = (
    data["anomaly"]
    .apply(
        lambda x: 80 if x == -1 else 20
    )
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_20720\1469810320.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["anomaly_score"] = (


In [38]:
data["graph_score"] = (
    data["degree"]
    .apply(
        lambda x:
        min(x*5,100)
    )
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_20720\2038323060.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["graph_score"] = (


In [39]:
data["rule_score"] = (

    data["high_amount_flag"] * 40

    +

    data["night_transaction"] * 30

    +

    data["transaction_velocity"]
    .apply(
        lambda x:
        30 if x > 10 else 0
    )

)

C:\Users\Hp\AppData\Local\Temp\ipykernel_20720\2719681811.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["rule_score"] = (


In [42]:
import sys
from pathlib import Path
project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src" / "final_risk_engine.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing src.")

sys.path.insert(0, str(project_root))

from src.final_risk_engine import calculate_final_risk, risk_level

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "src" / "final_risk_engine.py").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing src.")

sys.path.insert(0, str(project_root))


In [43]:
data["final_risk_score"] = data.apply(

lambda row:

calculate_final_risk(

row["ml_probability"],

row["anomaly_score"],

row["graph_score"],

row["rule_score"]

),

axis=1

)

C:\Users\Hp\AppData\Local\Temp\ipykernel_20720\1306364045.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["final_risk_score"] = data.apply(


In [46]:
data["final_risk_level"] = data["final_risk_score"].apply(risk_level)

data[
    [
        "txId",
        "final_risk_score",
        "final_risk_level",
    ]
].head(20)

,txId,final_risk_score,final_risk_level
0,0,NaN,LOW
1,1,NaN,LOW
2,2,NaN,LOW
3,3,NaN,LOW
4,4,NaN,LOW
5,5,NaN,LOW
6,6,NaN,LOW
7,7,NaN,LOW
8,8,NaN,LOW
9,9,NaN,LOW


In [47]:
data.to_csv(

"../data/processed/final_risk_results.csv",

index=False

)

In [48]:
def priority(score):

    if score >= 80:
        return "URGENT"

    elif score >= 50:
        return "HIGH"

    elif score >= 30:
        return "MEDIUM"

    else:
        return "LOW"


data["priority"] = data["final_risk_score"].apply(priority)

C:\Users\Hp\AppData\Local\Temp\ipykernel_20720\1630893191.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["priority"] = data["final_risk_score"].apply(priority)


In [49]:
data.to_csv(
"../data/processed/final_risk_results.csv",
index=False
)